![image_1780381321820.png](./image_1780381321820.png "image_1780381321820.png")

[Uploading image_1780381345100.png...]

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
# Initialize Spark session
spark = SparkSession.builder.appName("ShipmentsData").getOrCreate()

# Define the dataset
data = [
    (1, "2023-06-01", "2023-06-02", "2023-06-03"),
    (2, "2023-06-01", "2023-06-02", "2023-06-04"),
    (3, "2023-06-01", "2023-06-03", "2023-06-05"),
    (4, "2023-06-02", "2023-06-04", "2023-06-07"),
    (5, "2023-06-02", "2023-06-05", "2023-06-08"),
    (6, "2023-06-03", "2023-06-04", "2023-06-10"),
    (7, "2023-06-03", "2023-06-06", "2023-06-12"),
    (8, "2023-06-04", "2023-06-05", "2023-06-06"),
    (9, "2023-06-05", "2023-06-06", "2023-06-07"),
    (10, "2023-06-05", "2023-06-08", "2023-06-14")
]

# Define schema
columns = ["shipment_id", "order_date", "ship_date", "delivery_date"]

# Create DataFrame
shipments_df = spark.createDataFrame(data, columns)

# Show DataFrame
shipments_df.show()


In [0]:
result_df = (
    shipments_df
    .withColumn("delay_category",
        f.when((f.datediff("delivery_date","order_date") >= 0) & 
               (f.datediff("delivery_date","order_date") <= 3), "On Time")
        .when((f.datediff("delivery_date","order_date") > 3) & 
              (f.datediff("delivery_date","order_date") <= 7), "Slightly Delayed")
        .otherwise("Very Delayed"))
    .groupBy("delay_category")
    .agg(f.count("delay_category").alias("shipment_count"))
    .select(
        f.col("delay_category"),
        f.col("shipment_count")
    )
)
display(result_df)